# 見本の場面を Cycles で Drive に描く（colab-render の試し）

Colab の GPU で描いて、Google Drive に書き出すノートブック（colab-render で作った）。

| | |
|---|---|
| 場面一式 | ノートブックの中でコードで組み立てる（`demo_scene.py`。Drive に置く物は無い） |
| 描くコマ | 1〜120（120 コマを、60 コマずつ 2 回に分けて） |
| 書き出し先 | マイドライブ / `colab_render` / `out` / `demo_cycles` |
| 動画 | 描き終えたら、書き出し先に `demo_cycles.mp4` を作る（30fps） |
| 使う GPU | どれでも |
| Blender | 5.1.2（cycles） |

**使い方**
1. Drive に置く物はありません（場面はノートブックの中で組み立てます）。描いたコマは、Google Drive の `マイドライブ/colab_render/out/` に書き出します。
2. このノートブックを開いて、「ランタイム」→「ランタイムのタイプを変更」で GPU（ふだんは **L4**）を選ぶ。
3. 「ランタイム」→「すべてのセルを実行」。2 のセルで Google Drive へのアクセスを聞かれるので、許可する。**ここだけは、その場にいる必要があります。**
4. あとは離れてかまいません（ブラウザのタブは開いたまま、Mac はスリープしないように）。進み具合は、Drive の `out/demo_cycles/_colab_render/progress.txt` でも見られます。
5. 描き終わると（または先へ進めなくなると）、ランタイムは自分で手放されます。描いたコマは Drive に残ります。
6. 途中で切れたら、同じノートブックをもう一度「すべてのセルを実行」すると、描き終えた区切りは飛ばして、続きから描きます。

猿の頭 100 体の見本の場面（ノートブックの中でコードで組み立てる）を Cycles で 120 コマ描いて、4 秒の MP4 にします。Drive の許可のほかに、用意する物はありません。

## 1. この実行環境（GPU・CPU・メモリ）

In [ ]:
JOB = {'title': '見本の場面を Cycles で Drive に描く（colab-render の試し）', 'blender': '5.1.2', 'drive_dir': 'colab_render', 'build': '../scripts/demo_scene.py', 'build_args': '--out demo.blend --engine cycles --frames 120 --fps 30', 'engine': 'cycles', 'command': '-b --factory-startup demo.blend -P {GPU} -o {OUT}/f_#### -s {START} -e {END} -a', 'frames': [1, 120], 'chunk': 60, 'out': 'demo_cycles', 'expect_per_frame': 1, 'timeout_per_frame': 300, 'accept_gpus': [], 'release_at_end': True, 'env': {}, 'video': {'pattern': 'f_%04d.png', 'fps': 30, 'name': 'demo_cycles.mp4'}, 'note': '猿の頭 100 体の見本の場面（ノートブックの中でコードで組み立てる）を Cycles で 120 コマ描いて、4 秒の MP4 にします。Drive の許可のほかに、用意する物はありません。'}
import os, re, json, glob, shutil, subprocess, time
from google.colab import runtime

def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

def stop(msg, fail=True):
    """Print why, write everything out to Drive, give the runtime back (release_at_end), and stop a failed run."""
    print(msg)
    try:
        from google.colab import drive
        if os.path.ismount("/content/drive"):
            drive.flush_and_unmount()          # the last files reach Drive before the runtime goes
    except Exception as e:
        print("Drive:", e)
    if JOB.get("release_at_end", True):
        print("ランタイムを手放します。")
        runtime.unassign()
    if fail:
        raise SystemExit(msg)

info = {}
q = sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader")
if not q or "failed" in q.lower() or "not found" in q.lower():
    stop("GPU が見えません。ランタイムのタイプで GPU を選んでください。\n" + q)
info["gpu"], info["driver"], info["vram"] = [s.strip() for s in q.splitlines()[0].split(",")]
info["cpus"] = sh("nproc")
info["ram"] = sh("free -h | awk '/Mem/ {print $2}'")
for k, v in info.items():
    print(f"{k:8s} {v}")
ACCEPT = JOB.get("accept_gpus") or []
if ACCEPT and not any(a.lower() in info["gpu"].lower() for a in ACCEPT):
    stop(f"この GPU（{info['gpu']}）は、この仕事では使わない設定です（accept_gpus: {ACCEPT}）。")

## 2. Google Drive をつなぐ（アクセスの許可を聞かれます）

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
except Exception as e:
    stop(f"Google Drive をつなげませんでした: {e}")
BASE = f"/content/drive/MyDrive/{JOB['drive_dir']}"
OUT = f"{BASE}/out/{JOB['out']}"
if JOB.get("build"):
    print("場面一式: ノートブックの中で組み立てる")
else:
    BUNDLE = f"{BASE}/{JOB['bundle']}"
    if not os.path.exists(BUNDLE):
        stop(f"Drive に {JOB['drive_dir']}/{JOB['bundle']} が見つかりません。置いてから、もう一度実行してください。")
    print(f"場面一式: {BUNDLE}（{os.path.getsize(BUNDLE) / 1e6:.0f} MB）")
os.makedirs(OUT, exist_ok=True)
done = len(glob.glob(f"{OUT}/_colab_render/done_*"))
print(f"書き出し先: {OUT}" + (f"（前の回に描き終えた区切り: {done}）" if done else ""))

## 3. Blender（Linux 版）と、動かすのに要る部品を入れる

In [ ]:
V = JOB["blender"]
BL = f"/content/blender-{V}-linux-x64/blender"
if not os.path.exists(BL):
    mm = ".".join(V.split(".")[:2])
    sh(f"cd /content && wget -q https://download.blender.org/release/Blender{mm}/blender-{V}-linux-x64.tar.xz"
       f" && tar -xf blender-{V}-linux-x64.tar.xz && rm blender-{V}-linux-x64.tar.xz", check=True)
sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 "
   "libxrender1 libxkbcommon0 libsm6 libice6 libgl1 libegl1 libglvnd0 > /dev/null", check=True)
print(sh(f"{BL} --version | head -1"))

## 4. NVIDIA の部品を、ドライバーと同じ版で足す（無い物だけ。2026-11 に試した T4・L4・A100 では、はじめから入っていた）

In [ ]:
D = info["driver"]
dst, host = "/usr/lib/x86_64-linux-gnu", "/usr/lib64-nvidia"
WANT = ("libEGL_nvidia", "libGLX_nvidia", "libnvidia-eglcore", "libnvidia-glcore", "libnvidia-glsi", "libnvidia-tls",
        "libnvidia-glvkspirv", "libnvidia-gpucomp", "libnvoptix", "libnvidia-rtcore", "libnvidia-ptxjitcompiler")
have = lambda n: os.path.exists(f"{dst}/{n}") or os.path.exists(f"{host}/{n}")
NV = "はじめから入っていた"
if not (have(f"libEGL_nvidia.so.{D}") and have(f"libnvoptix.so.{D}")):
    run = f"NVIDIA-Linux-x86_64-{D}.run"
    got = None
    for url in (f"https://us.download.nvidia.com/tesla/{D}/{run}",
                f"https://download.nvidia.com/XFree86/Linux-x86_64/{D}/{run}"):
        if sh(f"cd /content && wget -q -O {run} {url} && echo OK").endswith("OK"):
            got = url
            break
    if not got:
        NV = f"ドライバー {D} の配布ファイルが見つからない"
    else:
        sh(f"cd /content && rm -rf nv && sh {run} --extract-only --target nv > /dev/null", check=True)
        put = [n for n in sorted(os.listdir("/content/nv")) if n.endswith("." + D) and n.startswith(WANT) and not have(n)]
        for n in put:
            sh(f"cp /content/nv/{n} {dst}/")
        if os.path.exists("/content/nv/nvoptix.bin") and not os.path.exists("/usr/share/nvidia/nvoptix.bin"):
            os.makedirs("/usr/share/nvidia", exist_ok=True)
            sh("cp /content/nv/nvoptix.bin /usr/share/nvidia/")
        if os.path.exists("/content/nv/10_nvidia.json"):
            os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
            sh("cp -n /content/nv/10_nvidia.json /usr/share/glvnd/egl_vendor.d/")
        NV = "足した: " + (", ".join(n.split(".so")[0] for n in put) or "なし")
for base, so in (("libEGL_nvidia.so", "0"), ("libGLX_nvidia.so", "0"), ("libnvoptix.so", "1"),
                 ("libnvidia-ptxjitcompiler.so", "1")):
    if os.path.exists(f"{dst}/{base}.{D}") and not have(f"{base}.{so}"):
        sh(f"ln -sf {base}.{D} {dst}/{base}.{so}")
J = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(J):
    os.makedirs(os.path.dirname(J), exist_ok=True)
    open(J, "w").write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
sh("ldconfig")
os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = J
print("NVIDIA の部品:", NV)

## 5. 場面一式を開いて、GPU で描けるかを確かめる

In [ ]:
GPU_PICK = '# SPDX-License-Identifier: GPL-3.0-or-later\n# Copyright (C) 2026 たぬ\n"""Make Cycles render on this machine\'s GPU, whatever the scripts after this one set: Metal on a Mac, OptiX (else CUDA)\non Linux with NVIDIA, such as Google Colab.  Put it before a project\'s own scripts:\n\n    blender -b scene.blend -P gpu_pick.py -P render.py -- ...\n    blender -b --factory-startup -P gpu_pick.py          (alone: only prints the GPU Cycles would use)\n\nWhy: scripts written on a Mac often set `prefs.compute_device_type = "METAL"`.  On Linux that raises TypeError, and a\nscript that catches it falls back to the CPU without a word (a 1080p Cycles frame then takes minutes, not seconds).\nThis script picks the GPU at once, and again right before every render (render_init / render_pre handlers, kept\nacross file loads), so a later script\'s fallback is undone at render time.  EEVEE needs nothing from here: it draws\nwith the GPU through OpenGL (EGL when there is no screen).\n\nPrints `GPU_PICK device <kind>: <names>` once, and `GPU_PICK fixed: ...` when it had to undo a CPU fallback (then the\ndenoiser goes to the GPU too, as the script would have set for a GPU).\n"""\nimport sys\n\nimport bpy\n\nKINDS = ("METAL",) if sys.platform == "darwin" else ("OPTIX", "CUDA", "HIP", "ONEAPI")\n_state = {"kind": None, "said": False}\n\n\ndef _say(line):\n    if not _state["said"]:\n        print(line, flush=True)\n        _state["said"] = True\n\n\ndef pick():\n    """Choose the first kind of GPU that has a device here; return the kind, or None (CPU only)."""\n    prefs = bpy.context.preferences.addons["cycles"].preferences\n    for kind in KINDS:\n        try:\n            prefs.compute_device_type = kind\n        except TypeError:                      # a kind this build or platform does not know\n            continue\n        prefs.get_devices()\n        gpus = [d for d in prefs.devices if d.type == kind]\n        if not gpus:\n            continue\n        for d in prefs.devices:\n            d.use = d.type == kind\n        _state["kind"] = kind\n        _say(f"GPU_PICK device {kind}: {\', \'.join(d.name for d in gpus)}")\n        return kind\n    _state["kind"] = None\n    _say("GPU_PICK device CPU (no GPU that Cycles can use)")\n    return None\n\n\n@bpy.app.handlers.persistent\ndef _before_render(scene, *_):\n    if scene.render.engine != "CYCLES":\n        return\n    prefs = bpy.context.preferences.addons["cycles"].preferences\n    kind = _state["kind"] if prefs.compute_device_type == _state["kind"] else pick()\n    if kind and scene.cycles.device != "GPU":\n        scene.cycles.device = "GPU"\n        scene.cycles.denoising_use_gpu = True\n        print(f"GPU_PICK fixed: scene \'{scene.name}\' was set to render on the CPU; rendering on {kind}", flush=True)\n\n\npick()\nfor _h in (bpy.app.handlers.render_init, bpy.app.handlers.render_pre):\n    if _before_render not in _h:\n        _h.append(_before_render)\n'
RUNNER = '# SPDX-License-Identifier: GPL-3.0-or-later\n# Copyright (C) 2026 たぬ\n"""Render a frame range in chunks, one Blender per chunk; copy each finished chunk to an output folder (Google Drive\non Colab) and mark it done, so that a run that stopped (a dropped runtime, the session limit) carries on where it\nstopped.  Standard library only: Colab runs it with its own Python, and it runs the same on a Mac (to try a job before\nit goes to Colab, or to render the Mac\'s share of a job).\n\n    python3 colab_job.py --job job.json --blender BL --root BUNDLE_DIR --out OUT_DIR [--gpu-pick gpu_pick.py]\n    python3 colab_job.py --job job.json --blender BL --check [--gpu-pick gpu_pick.py]\n        (--check only: can Cycles / EEVEE draw with the GPU here?  Exit 0 if the job\'s engine can)\n\njob.json (an example: assets/job_example.json):\n    command    Blender\'s arguments, with {GPU} (gpu_pick.py), {START} and {END} (the chunk\'s first and last frame),\n               {OUT} (an empty folder for this chunk: everything written under it is copied to OUT_DIR, sub-folders\n               kept) and {ROOT} (the bundle\'s folder; Blender runs there).  Write {{ and }} for a literal brace.\n    frames     [first, last]          chunk  frames a Blender renders (default 100)\n    engine     cycles | eevee | both  (what --check asks for)\n    expect_per_frame   files a frame writes under {OUT} (default 1; 0 = do not count)\n    timeout_per_frame  seconds (default 300; a chunk gets 600 + frames x this)\n    env        extra environment variables for Blender (values may use {ROOT})\n    workers    Blenders that render at the same time, each a chunk of its own (default 1).  For a scene whose frames\n               wait on the CPU (heavy scene evaluation, Python that places things every frame) rather than on the\n               GPU: one GPU then feeds several Blenders.  Each needs its own share of the GPU\'s and the machine\'s\n               memory; the summary\'s gpu_util_mean, gpu_mem_peak_mb, ram_peak_gb and load_mean tell whether there is\n               room (too many for the machine\'s memory and Colab ends the runtime without a word)\n    ram_limit  (with workers > 1; default 0.85) the share of the machine\'s memory in use at which the youngest\n               Blender is ended and the job goes on with one Blender fewer; its chunk is rendered again when there\n               is room, and it does not count as a failed try.  Linux only (/proc/meminfo)\n    echo       (optional) a regular expression: the lines of a finished chunk\'s log that match it are printed\n               (the last 8), e.g. the render script\'s own timing lines\n    build_args (with --build-script) the arguments of the script that makes the scene in ROOT before the first\n               chunk (for a scene made by code instead of a bundle; values may use {ROOT})\n    video      (optional) make an MP4 of the frames once every chunk is done, with ffmpeg:\n               {"pattern": "f_%04d.png" (under OUT_DIR), "fps": 60, "name": "<file>.mp4" (into OUT_DIR),\n                "start": first frame (default frames[0]), "crf": 15, "preset": "slow", "refs": 4, "level": "4.2",\n                "audio": a file in the bundle (optional)}.  Colour tags: BT.709 primaries and matrix with the sRGB\n               transfer curve, so colour-managed players on macOS show it like the PNG frames (a bt709 transfer tag\n               makes them 6-7/255 lighter).  Skipped, with a line saying so, when a frame of the range is missing.\n\n    python3 colab_job.py --job job.json --root BUNDLE_DIR --out OUT_DIR --video-only    (the MP4 of what is there)\n\nState lives in OUT_DIR/_colab_render/: done_<first>_<last> for each finished chunk (it is skipped next time),\nprogress.txt (one line per chunk, with the time left), logs/<first>_<last>.log, summary.json, watch.txt (a line a\nminute while chunks render: load, memory in use, the GPU\'s utilisation and memory).\nExit 0 when every chunk is done (and the MP4, if asked for, is made), 1 when a chunk failed twice or the MP4 could\nnot be made, 2 when the GPU check failed.\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport shlex\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nEEVEE_PROBE = ("import bpy, gpu; sc = bpy.context.scene; sc.render.engine = \'BLENDER_EEVEE\'; "\n               "sc.render.resolution_percentage = 10; sc.eevee.taa_render_samples = 1; "\n               "sc.render.filepath = {path!r}; bpy.ops.render.render(write_still=True); "\n               "print(\'EEVEE_PROBE\', gpu.platform.vendor_get(), \'|\', gpu.platform.renderer_get(), \'|\', "\n               "gpu.platform.version_get())")\n\n\ndef say(line, progress=None):\n    print(line, flush=True)\n    if progress:\n        with open(progress, "a", encoding="utf-8") as fh:\n            fh.write(time.strftime("%Y-%m-%d %H:%M:%S ") + line + "\\n")\n\n\ndef hms(s):\n    s = int(round(s))\n    return f"{s // 3600}:{s // 60 % 60:02d}:{s % 60:02d}"\n\n\ndef check(a, job):\n    """Can the engine(s) of the job draw with the GPU on this machine?"""\n    engine = job.get("engine", "cycles")\n    ok = True\n    if engine in ("cycles", "both"):\n        out = subprocess.run([a.blender, "-b", "--factory-startup", "-P", a.gpu_pick],\n                             capture_output=True, text=True, timeout=600).stdout\n        m = re.search(r"GPU_PICK device ([^\\n]+)", out)\n        dev = m.group(1).strip() if m else "?"\n        good = bool(m) and not dev.startswith("CPU")\n        say(f"CHECK cycles: {dev} -> {\'GPU\' if good else \'NOT ON THE GPU\'}")\n        ok &= good\n    if engine in ("eevee", "both"):\n        tmp = tempfile.mkdtemp(prefix="eevee_probe_")\n        expr = EEVEE_PROBE.format(path=os.path.join(tmp, "probe.png"))\n        out = subprocess.run([a.blender, "-b", "--factory-startup", "--python-expr", expr],\n                             capture_output=True, text=True, timeout=600).stdout\n        m = re.search(r"EEVEE_PROBE ([^\\n]+)", out)\n        dev = m.group(1).strip() if m else "?"\n        good = bool(m) and not any(w in dev.lower() for w in ("llvmpipe", "softpipe", "software"))\n        say(f"CHECK eevee: {dev} -> {\'GPU\' if good else \'NOT ON THE GPU\'}")\n        ok &= good\n    return ok\n\n\ndef chunks(job):\n    first, last = job["frames"]\n    n = int(job.get("chunk", 100))\n    return [(s, min(s + n - 1, last)) for s in range(first, last + 1, n)]\n\n\ndef files_under(folder):\n    return [os.path.join(d, f) for d, _, fs in os.walk(folder) for f in fs]\n\n\ndef run_chunk(a, job, s, e, work, log, running=None):\n    """running: a dict that holds (the Blender\'s process, when it started) under (s, e) while it runs."""\n    shutil.rmtree(work, ignore_errors=True)\n    os.makedirs(work)\n    fill = {"GPU": a.gpu_pick, "START": s, "END": e, "OUT": work, "ROOT": a.root}\n    cmd = [a.blender] + shlex.split(job["command"].format(**fill))\n    env = dict(os.environ)\n    env.update({k: str(v).format(ROOT=a.root) for k, v in job.get("env", {}).items()})\n    limit = 600 + (e - s + 1) * int(job.get("timeout_per_frame", 300))\n    t0 = time.time()\n    with open(log, "w", encoding="utf-8") as fh:\n        fh.write("$ " + shlex.join(cmd) + "\\n")\n        fh.flush()\n        proc = subprocess.Popen(cmd, cwd=a.root, env=env, stdout=fh, stderr=subprocess.STDOUT)\n        if running is not None:\n            running[(s, e)] = (proc, t0)\n        try:\n            rc = proc.wait(timeout=limit)\n        except subprocess.TimeoutExpired:\n            proc.kill()\n            proc.wait()\n            rc = f"timeout after {limit} s"\n        finally:\n            if running is not None:\n                running.pop((s, e), None)\n    secs = time.time() - t0\n    made = files_under(work)\n    want = (e - s + 1) * int(job.get("expect_per_frame", 1))\n    problem = None\n    if rc != 0:\n        problem = f"Blender ended with {rc}"\n    elif len(made) < want:\n        problem = f"{len(made)} file(s) written, {want} expected"\n    return made, secs, problem\n\n\ndef tail(path, n=25):\n    with open(path, encoding="utf-8", errors="replace") as fh:\n        return "".join(fh.readlines()[-n:])\n\n\ndef echo(path, pattern, n=8):\n    """The last lines of a chunk\'s log that match the job\'s "echo"."""\n    with open(path, encoding="utf-8", errors="replace") as fh:\n        return [l.rstrip() for l in fh if re.search(pattern, l)][-n:]\n\n\ndef ram_used():\n    """(GB in use, GB in all) from /proc/meminfo (Linux); None elsewhere."""\n    try:\n        kb = {}\n        with open("/proc/meminfo") as fh:\n            for line in fh:\n                k, v = line.split(":", 1)\n                kb[k] = float(v.split()[0])\n        return (kb["MemTotal"] - kb["MemAvailable"]) / 1e6, kb["MemTotal"] / 1e6\n    except (OSError, KeyError, ValueError, IndexError):\n        return None\n\n\ndef watch(done, got, every=10.0, trace=None, on_ram=None):\n    """While the chunks render: how busy the machine and the GPU are (the load average, the machine\'s memory in use;\n    with nvidia-smi the GPU\'s utilisation and memory).  It tells a scene that waits on the CPU from one that waits on\n    the GPU, and whether another Blender would fit.  trace: a file that gets a line every minute (in OUT_DIR, so that\n    after a runtime that died there is something to read: what was in use shortly before).  on_ram(GB in use, GB in\n    all): called with every reading of the memory (main()\'s guard)."""\n    smi = shutil.which("nvidia-smi")\n    k = 0\n    while not done.wait(every):\n        k += 1\n        line = time.strftime("%Y-%m-%d %H:%M:%S")\n        try:\n            load = os.getloadavg()[0]\n            got["load"] += load\n            got["n"] += 1\n            line += f" load {load:.1f}"\n        except OSError:\n            pass\n        r = ram_used()\n        if r:\n            got["ram"] = max(got.get("ram", 0.0), r[0])\n            got["ram_total"] = r[1]\n            line += f" ram {r[0]:.1f}/{r[1]:.1f} GB"\n            if on_ram:\n                try:\n                    on_ram(*r)\n                except Exception as ex:                      # the watcher must outlive its guard\n                    print(f"(memory guard: {ex})", flush=True)\n        if smi:\n            try:\n                out = subprocess.run([smi, "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],\n                                     capture_output=True, text=True, timeout=20).stdout\n                u, m = [float(x) for x in out.splitlines()[0].split(",")]\n                got["gpu_n"] += 1\n                got["util"] += u\n                got["mem"] = max(got["mem"], m)\n                line += f" gpu {u:.0f}% {m:.0f} MB"\n            except Exception:\n                pass\n        if trace and k % max(1, round(60 / every)) == 0:\n            try:\n                with open(trace, "a", encoding="utf-8") as fh:\n                    fh.write(line + "\\n")\n            except OSError:\n                pass\n\n\ndef make_video(a, job, progress):\n    """The MP4 of the job\'s frames in OUT_DIR (job["video"]); returns (path or None, problem or None)."""\n    v = job.get("video")\n    if not v:\n        return None, None\n    first, last = job["frames"]\n    start = int(v.get("start", first))\n    count = last - first + 1\n    pattern = os.path.join(a.out, v["pattern"])\n    missing = [i for i in range(start, start + count) if not os.path.exists(pattern % i)]\n    if missing:\n        return None, f"{len(missing)} frame(s) missing (first {pattern % missing[0]})"\n    name = v.get("name") or (os.path.basename(os.path.normpath(a.out)) + ".mp4")\n    tmp = os.path.join(tempfile.mkdtemp(prefix="colab_video_"), name)\n    color = ["-vf", "scale=in_range=full:out_range=tv:out_color_matrix=bt709,format=yuv420p,"\n                    "setparams=color_primaries=bt709:color_trc=iec61966-2-1:colorspace=bt709:range=tv",\n             "-colorspace", "bt709", "-color_primaries", "bt709", "-color_trc", "iec61966-2-1"]\n    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(v.get("fps", 60)), "-start_number", str(start),\n           "-i", pattern]\n    if v.get("audio"):\n        cmd += ["-i", os.path.join(a.root, v["audio"]), "-map", "0:v", "-map", "1:a", "-c:a", "aac", "-b:a", "192k",\n                "-shortest"]\n    cmd += ["-frames:v", str(count), "-c:v", "libx264", "-preset", str(v.get("preset", "slow")),\n            "-crf", str(v.get("crf", 15)), "-refs", str(v.get("refs", 4))]\n    if v.get("level", "4.2"):\n        cmd += ["-level:v", str(v.get("level", "4.2"))]\n    cmd += color + ["-movflags", "+faststart", tmp]\n    say(f"VIDEO making {name} from {count} frame(s) at {v.get(\'fps\', 60)} fps", progress)\n    t0 = time.time()\n    r = subprocess.run(cmd, capture_output=True, text=True)\n    if r.returncode != 0 or not os.path.exists(tmp):\n        return None, "ffmpeg: " + (r.stderr.strip()[-800:] or f"exit {r.returncode}")\n    dest = os.path.join(a.out, name)\n    shutil.copyfile(tmp, dest)\n    say(f"VIDEO written {dest} ({os.path.getsize(dest) / 1e6:.1f} MB, {time.time() - t0:.0f} s)", progress)\n    return dest, None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--job", required=True)\n    ap.add_argument("--blender", default="", help="the Blender to run (not needed with --video-only)")\n    ap.add_argument("--gpu-pick", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "gpu_pick.py"))\n    ap.add_argument("--root", default=".")\n    ap.add_argument("--out", default="")\n    ap.add_argument("--work", default="", help="the folder a chunk renders into (default: a temporary one)")\n    ap.add_argument("--check", action="store_true")\n    ap.add_argument("--video-only", action="store_true", help="only make the MP4 of the frames already in --out")\n    ap.add_argument("--build-script", default="", help="a Blender script that makes the scene in --root first")\n    a = ap.parse_args()\n    job = json.load(open(a.job, encoding="utf-8"))\n    a.root = os.path.abspath(a.root)\n    a.gpu_pick = os.path.abspath(a.gpu_pick)\n    if not a.blender and not a.video_only:\n        sys.exit("--blender is needed")\n    if a.check:\n        sys.exit(0 if check(a, job) else 2)\n    if not a.out:\n        sys.exit("--out is needed")\n    state = os.path.join(a.out, "_colab_render")\n    os.makedirs(os.path.join(state, "logs"), exist_ok=True)\n    progress = os.path.join(state, "progress.txt")\n    if a.video_only:\n        path, problem = make_video(a, job, progress)\n        if problem or not path:\n            say(f"VIDEO not made: {problem or \'no video in the job\'}", progress)\n            sys.exit(1)\n        return\n    work = a.work or tempfile.mkdtemp(prefix="colab_chunk_")\n    logdir = tempfile.mkdtemp(prefix="colab_logs_")\n    if a.build_script:\n        os.makedirs(a.root, exist_ok=True)\n        cmd = [a.blender, "-b", "--factory-startup", "-P", os.path.abspath(a.build_script), "--"] + \\\n            shlex.split(str(job.get("build_args", "")).format(ROOT=a.root))\n        r = subprocess.run(cmd, cwd=a.root, capture_output=True, text=True, timeout=1800)\n        lines = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]\n        if r.returncode != 0:\n            say(f"BUILD failed (exit {r.returncode})", progress)\n            print("\\n".join(lines[-25:]), flush=True)\n            sys.exit(1)\n        said = [l for l in lines if re.search(r"\\bsaved\\b", l, re.I)]\n        say(f"BUILD done: {(said or lines or [\'\'])[-1].strip()}", progress)\n    todo = chunks(job)\n    workers = max(1, int(job.get("workers", 1)))\n    left = sum(e - s + 1 for s, e in todo if not os.path.exists(os.path.join(state, f"done_{s}_{e}")))\n    say(f"START {job.get(\'title\', \'\')}: frames {job[\'frames\'][0]}-{job[\'frames\'][1]} in {len(todo)} chunk(s); "\n        f"{left} frame(s) to render" + (f", {workers} Blenders at a time" if workers > 1 else ""), progress)\n    st = {"t": 0.0, "f": 0, "left": left, "device": "", "fail": "", "shed": 0}\n    lock = threading.Lock()\n    t_wall = time.time()\n    got = {"n": 0, "load": 0.0, "gpu_n": 0, "util": 0.0, "mem": 0.0}\n    running, shed = {}, set()\n    slots = {"max": workers, "busy": 0, "quiet": 0.0}\n    room = threading.Condition()\n    ram_limit = float(job.get("ram_limit", 0.85))\n\n    def guard(used, total):\n        """The machine\'s memory is nearly full while several Blenders run: end the youngest one and go on with one\n        Blender fewer (its chunk is rendered again when there is room).  Colab ends a runtime whose memory is full\n        without a word, and everything in flight is lost."""\n        if used < ram_limit * total or time.time() < slots["quiet"]:\n            return\n        with room:\n            now = list(running.items())\n            if len(now) < 2:\n                return\n            key, (proc, _) = max(now, key=lambda kv: kv[1][1])\n            shed.add(key)\n            slots["max"] = max(1, len(now) - 1)\n            slots["quiet"] = time.time() + 60\n            st["shed"] += 1\n            proc.kill()\n        with lock:\n            say(f"MEMORY {used:.1f} of {total:.1f} GB in use with {len(now)} Blenders: ended {key[0]}-{key[1]} "\n                f"(it is rendered again later), {slots[\'max\']} at a time from here", progress)\n\n    over = threading.Event()\n    threading.Thread(target=watch, args=(over, got),\n                     kwargs={"trace": os.path.join(state, "watch.txt"), "on_ram": guard}, daemon=True).start()\n\n    def extra():\n        x = {}\n        if workers > 1:\n            x.update(workers=workers, wall_seconds=round(time.time() - t_wall, 1),\n                     s_per_frame_each=round(st["t"] / st["f"], 2) if st["f"] else None)\n        if got["n"]:\n            x["load_mean"] = round(got["load"] / got["n"], 1)\n        if got["gpu_n"]:\n            x.update(gpu_util_mean=round(got["util"] / got["gpu_n"]), gpu_mem_peak_mb=round(got["mem"]))\n        if got.get("ram"):\n            x.update(ram_peak_gb=round(got["ram"], 1), ram_total_gb=round(got["ram_total"], 1))\n        if st["shed"]:\n            x.update(memory_guard=st["shed"], workers_at_end=slots["max"])\n        return x\n\n    def one(s, e):\n        """Render a chunk (twice if the first try fails), copy what it wrote to OUT_DIR, mark it done."""\n        if st["fail"]:\n            return\n        wdir = work if workers == 1 else os.path.join(work, f"{s}_{e}")\n        log = os.path.join(logdir, f"{s}_{e}.log")\n        attempt = 0\n        while True:\n            with room:                                   # a place among the Blenders (the guard may have taken one away)\n                while slots["busy"] >= slots["max"]:\n                    room.wait()\n                slots["busy"] += 1\n            try:\n                if st["fail"]:\n                    return\n                made, secs, problem = run_chunk(a, job, s, e, wdir, log, running)\n            finally:\n                with room:\n                    slots["busy"] -= 1\n                    room.notify_all()\n            shutil.copyfile(log, os.path.join(state, "logs", f"{s}_{e}.log"))\n            if (s, e) in shed:\n                shed.discard((s, e))\n                if problem:                              # ended by the guard: not a failed try\n                    continue\n            attempt += 1\n            if not problem:\n                break\n            with lock:\n                say(f"FAIL {s}-{e} (try {attempt}): {problem}", progress)\n                print(tail(log), flush=True)\n            if attempt == 2:\n                break\n        if problem:\n            with lock:\n                st["fail"] = st["fail"] or f"{s}-{e}: {problem}"\n            return\n        for f in made:\n            dest = os.path.join(a.out, os.path.relpath(f, wdir))\n            os.makedirs(os.path.dirname(dest), exist_ok=True)\n            shutil.copyfile(f, dest)\n        with open(os.path.join(state, f"done_{s}_{e}"), "w", encoding="utf-8") as fh:\n            fh.write(f"{time.strftime(\'%Y-%m-%d %H:%M:%S\')} {secs:.1f} s {len(made)} file(s)\\n")\n        with lock:\n            if not st["device"]:\n                if job.get("engine", "cycles") == "eevee":   # gpu_pick.py names the Cycles device; EEVEE draws with OpenGL\n                    st["device"] = "OpenGL (EEVEE)"\n                else:\n                    m = re.search(r"GPU_PICK device ([^\\n]+)", open(log, encoding="utf-8", errors="replace").read())\n                    st["device"] = m.group(1).strip() if m else ""\n            n = e - s + 1\n            st["t"], st["f"], st["left"] = st["t"] + secs, st["f"] + n, st["left"] - n\n            rate = (time.time() - t_wall if workers > 1 else st["t"]) / st["f"]\n            say(f"done {s}-{e}: {n} frame(s) in {hms(secs)} ({secs / n:.2f} s a frame with Blender\'s start), "\n                f"{st[\'left\']} left, about {hms(st[\'left\'] * rate)} to go", progress)\n            if job.get("echo"):\n                for line in echo(log, job["echo"]):\n                    print("  | " + line, flush=True)\n\n    skipped, rest = 0, []\n    for s, e in todo:\n        if os.path.exists(os.path.join(state, f"done_{s}_{e}")):\n            skipped += 1\n            say(f"skip {s}-{e} (done before)")\n        else:\n            rest.append((s, e))\n    if workers == 1:\n        for s, e in rest:\n            one(s, e)\n            if st["fail"]:\n                break\n    else:\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            for r in [pool.submit(one, s, e) for s, e in rest]:\n                r.result()\n    over.set()\n    t_run, f_run, device = st["t"], st["f"], st["device"]\n    if st["fail"]:\n        summary(state, job, todo, skipped, t_run, f_run, device, "failed", st["fail"], extra=extra())\n        sys.exit(1)\n    if workers > 1:\n        wall = time.time() - t_wall\n        say(f"ALL DONE: {f_run} frame(s) rendered now, {skipped} chunk(s) done before; "\n            f"{(wall / f_run if f_run else 0):.2f} s a frame with {workers} Blenders at a time "\n            f"({(t_run / f_run if f_run else 0):.2f} s a frame in each); output in {a.out}", progress)\n    else:\n        say(f"ALL DONE: {f_run} frame(s) rendered now, {skipped} chunk(s) done before; "\n            f"{(t_run / f_run if f_run else 0):.2f} s a frame; output in {a.out}", progress)\n    video, problem = make_video(a, job, progress)\n    if problem:\n        say(f"VIDEO not made: {problem}", progress)\n    summary(state, job, todo, skipped, t_run, f_run, device, "failed" if problem else "done",\n            f"video: {problem}" if problem else "", video, extra=extra())\n    if problem:\n        sys.exit(1)\n\n\ndef summary(state, job, todo, skipped, t_run, f_run, device, status, error, video=None, extra=None):\n    """seconds_now and s_per_frame: of the Blenders\' own times, one after another; with several workers of the\n    wall clock (extra["wall_seconds"]), and s_per_frame_each is what a frame took inside one Blender."""\n    extra = extra or {}\n    secs = extra.get("wall_seconds", t_run)\n    with open(os.path.join(state, "summary.json"), "w", encoding="utf-8") as fh:\n        json.dump({"title": job.get("title", ""), "status": status, "error": error,\n                   "engine": job.get("engine", "cycles"), "frames": job["frames"],\n                   "chunks": len(todo), "skipped_chunks": skipped, "frames_rendered_now": f_run,\n                   "seconds_now": round(secs, 1), "s_per_frame": round(secs / f_run, 2) if f_run else None,\n                   "device": device, "video": video, "finished": time.strftime("%Y-%m-%d %H:%M:%S"),\n                   **{k: v for k, v in extra.items() if k != "wall_seconds"}},\n                  fh, ensure_ascii=False, indent=1)\n\n\nif __name__ == "__main__":\n    main()\n'
BUILD = '# SPDX-License-Identifier: GPL-3.0-or-later\n# Copyright (C) 2026 たぬ\n"""Build a small sample scene and save it as a .blend, so the whole flow (render in chunks onto Drive, then an MP4)\ncan be tried without a scene of one\'s own.  Everything is made here: a grid of subdivided monkeys in five\nmaterials turning slowly, a glossy floor, a sun and eight coloured point lights with shadows, and a camera that\ncircles the scene.\n\n    blender -b --factory-startup -P demo_scene.py -- --out demo.blend [--engine eevee|cycles] [--frames 120]\n            [--fps 30] [--res 1920x1080] [--samples 32] [--grid 10]\n\nThe .blend keeps the Subdivision modifier (not applied), so the file stays small; render it as an animation:\n    blender -b demo.blend -P gpu_pick.py -o out/f_#### -s 1 -e 120 -a\n"""\nimport argparse\nimport math\nimport os\nimport sys\n\nimport bpy\nfrom mathutils import Vector\n\n\ndef args():\n    argv = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--engine", default="eevee", choices=("eevee", "cycles"))\n    ap.add_argument("--frames", type=int, default=120)\n    ap.add_argument("--fps", type=int, default=30)\n    ap.add_argument("--res", default="1920x1080")\n    ap.add_argument("--samples", type=int, default=32)\n    ap.add_argument("--grid", type=int, default=10)\n    return ap.parse_args(argv)\n\n\ndef material(name, rgb, rough, metal=0.0):\n    m = bpy.data.materials.new(name)\n    m.use_nodes = True\n    p = next(n for n in m.node_tree.nodes if n.type == "BSDF_PRINCIPLED")\n    p.inputs["Base Color"].default_value = (*rgb, 1.0)\n    p.inputs["Roughness"].default_value = rough\n    p.inputs["Metallic"].default_value = metal\n    return m\n\n\ndef build(a):\n    for ob in list(bpy.data.objects):\n        bpy.data.objects.remove(ob, do_unlink=True)\n    sc = bpy.context.scene\n    sc.frame_start, sc.frame_end = 1, a.frames\n    sc.render.fps = a.fps\n    bpy.ops.mesh.primitive_monkey_add(size=0.8)\n    src = bpy.context.active_object\n    me = src.data\n    bpy.data.objects.remove(src, do_unlink=True)\n    mats = [material(f"m{i}", c, r, mt) for i, (c, r, mt) in enumerate((\n        ((0.8, 0.2, 0.15), 0.35, 0.0), ((0.15, 0.5, 0.8), 0.2, 0.0), ((0.9, 0.75, 0.3), 0.25, 1.0),\n        ((0.2, 0.7, 0.3), 0.6, 0.0), ((0.85, 0.85, 0.85), 0.1, 0.0)))]\n    me.materials.append(mats[0])               # one slot on the shared mesh; each object fills it with its own\n    n, half = a.grid, (a.grid - 1) / 2\n    for i in range(n):\n        for j in range(n):\n            ob = bpy.data.objects.new(f"monkey_{i}_{j}", me)\n            ob.location = ((i - half) * 1.6, (j - half) * 1.6, 0.55)\n            sc.collection.objects.link(ob)\n            mod = ob.modifiers.new("sub", "SUBSURF")\n            mod.levels = mod.render_levels = 2\n            ob.material_slots[0].link = "OBJECT"\n            ob.material_slots[0].material = mats[(i + j) % len(mats)]\n            start = math.radians((i * 37 + j * 53) % 360)\n            for f, turn in ((1, 0.0), (a.frames, math.radians(90 if (i + j) % 2 else -90))):\n                ob.rotation_euler = (0.0, 0.0, start + turn)\n                ob.keyframe_insert("rotation_euler", index=2, frame=f)\n    bpy.ops.mesh.primitive_plane_add(size=400.0)    # wide enough that its edge stays out of the picture\n    bpy.context.active_object.data.materials.append(material("floor", (0.3, 0.3, 0.32), 0.15))\n    sun = bpy.data.objects.new("sun", bpy.data.lights.new("sun", "SUN"))\n    sun.data.energy, sun.data.angle = 3.0, math.radians(2.0)\n    sun.rotation_euler = (math.radians(50), 0.0, math.radians(30))\n    sc.collection.objects.link(sun)\n    for k in range(8):\n        t = k / 8.0 * 2.0 * math.pi\n        lamp = bpy.data.objects.new(f"pt{k}", bpy.data.lights.new(f"pt{k}", "POINT"))\n        lamp.data.energy, lamp.data.shadow_soft_size = 400.0, 0.3\n        lamp.data.color = (1.0, 0.6 + 0.4 * (k % 2), 0.5 + 0.5 * ((k + 1) % 2))\n        lamp.location = (7.0 * math.cos(t), 7.0 * math.sin(t), 3.0)\n        sc.collection.objects.link(lamp)\n    cam = bpy.data.objects.new("cam", bpy.data.cameras.new("cam"))\n    sc.collection.objects.link(cam)\n    sc.camera = cam\n    cam.data.lens = 30.0\n    steps = 12\n    target = Vector((0.0, 0.0, 0.6))\n    for s in range(steps + 1):                 # a quarter turn round the scene, keyed every few frames\n        u = s / steps\n        f = 1 + round(u * (a.frames - 1))\n        t = math.radians(-20 + 90 * u)\n        cam.location = (13.0 * math.cos(t), 13.0 * math.sin(t), 4.8 - 1.4 * u)\n        cam.rotation_euler = (target - cam.location).to_track_quat("-Z", "Y").to_euler()\n        cam.keyframe_insert("location", frame=f)\n        cam.keyframe_insert("rotation_euler", frame=f)\n    if sc.world is None:\n        sc.world = bpy.data.worlds.new("world")\n    sc.world.use_nodes = True\n    bg = next(n for n in sc.world.node_tree.nodes if n.type == "BACKGROUND")\n    bg.inputs["Color"].default_value = (0.05, 0.06, 0.09, 1.0)\n    r = sc.render\n    w, h = (int(v) for v in a.res.split("x"))\n    r.resolution_x, r.resolution_y, r.resolution_percentage = w, h, 100\n    r.image_settings.file_format = "PNG"\n    r.image_settings.color_mode = "RGB"\n    r.use_stamp_filename = False\n    r.use_stamp_hostname = False\n    if a.engine == "eevee":\n        r.engine = "BLENDER_EEVEE"\n        e = sc.eevee\n        e.taa_render_samples = a.samples\n        for attr, val in (("use_raytracing", True), ("use_shadows", True), ("shadow_pool_size", "1024")):\n            try:                                # (the default 512 MB shadow pool overflowed: "Shadow buffer full")\n                setattr(e, attr, val)\n            except Exception as ex:\n                print("DEMO note:", attr, ex)\n    else:\n        r.engine = "CYCLES"\n        c = sc.cycles\n        c.device = "GPU"                        # gpu_pick.py chooses which GPU when it renders\n        c.samples = a.samples\n        c.use_adaptive_sampling = True\n        c.adaptive_threshold = 0.02\n        c.use_denoising = True\n        c.denoising_use_gpu = True\n    return n * n\n\n\ndef main():\n    a = args()\n    count = build(a)\n    out = os.path.abspath(a.out)\n    os.makedirs(os.path.dirname(out), exist_ok=True)\n    bpy.ops.wm.save_as_mainfile(filepath=out, compress=True)\n    print(f"DEMO saved {out}: {count} monkeys, {a.engine}, {a.frames} frames at {a.fps} fps, {a.res}, "\n          f"{a.samples} samples", flush=True)\n\n\nmain()\n'
open("/content/gpu_pick.py", "w").write(GPU_PICK)
open("/content/colab_job.py", "w").write(RUNNER)
json.dump(JOB, open("/content/job.json", "w"), ensure_ascii=False)
sh("rm -rf /content/job && mkdir -p /content/job", check=True)
if BUILD:                                   # the scene is made by code in the next cell, before the first chunk
    open("/content/job/build.py", "w").write(BUILD)
    print("場面を組み立てるスクリプトを置いた: /content/job/build.py")
else:
    shutil.copyfile(BUNDLE, "/content/bundle.zip")
    sh("cd /content/job && unzip -q /content/bundle.zip", check=True)
ROOT = os.path.join("/content/job", JOB.get("workdir", ""))
print("場面一式の場所:", ROOT)
r = subprocess.run(["python3", "/content/colab_job.py", "--job", "/content/job.json", "--blender", BL,
                    "--gpu-pick", "/content/gpu_pick.py", "--check"], capture_output=True, text=True)
print(r.stdout + r.stderr)
if r.returncode != 0:
    stop("GPU で描けないので、描かずに止めます（上の CHECK の行を Claude に見せてください）。")

## 6. 描く（区切りごとに Drive へ写す。終わったら、ランタイムを手放す）

In [ ]:
cmd = ["python3", "/content/colab_job.py", "--job", "/content/job.json", "--blender", BL, "--gpu-pick",
       "/content/gpu_pick.py", "--root", ROOT, "--out", OUT, "--work", "/content/chunk"]
if BUILD:
    cmd += ["--build-script", "/content/job/build.py"]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
except KeyboardInterrupt:          # stopped by hand: stop Blender, keep the runtime (the person is here)
    p.terminate()
    raise
summary = {}
try:
    summary = json.load(open(f"{OUT}/_colab_render/summary.json"))
except Exception as e:
    print("summary:", e)
print("\nまとめ: " + json.dumps({"gpu": info["gpu"], "cpus": info["cpus"], **summary}, ensure_ascii=False))
if rc == 0:
    stop("描き終えました。", fail=False)
else:
    stop(f"途中で止まりました（{summary.get('error', rc)}）。ログは Drive の _colab_render/logs/ にあります。")